# 01. QuartzNet 10x4 — обучение char-vocab ASR

Self-contained ноутбук: от сырых данных до submission.
Архитектура: QuartzNet-10x4 (TCS-свёрточный энкодер, ~4M параметров).
HP Random Search оборачивает весь цикл обучения.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [1]:
import os
from pathlib import Path

DATA_ROOT = Path("../notebooks/data")

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    import gdown
    import zipfile

    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_ROOT / "data.zip"
    if not zip_path.exists():
        gdown.download(
            url="https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link",
            output=str(zip_path),
            quiet=False,
            fuzzy=True,
        )
    zipfile.ZipFile(zip_path).extractall(DATA_ROOT)

## Пути (заполните вручную)

Задай абсолютные пути под свою среду. Все `FILL_ME_IN` должны быть реальными путями к train/dev/test и директории чекпоинтов.

In [2]:
from pathlib import Path

# DATA_ROOT was defined in the download cell above.
TRAIN_ROOT = DATA_ROOT / "data" / "train"
DEV_ROOT = DATA_ROOT / "data" / "dev"
TEST_ROOT: Path | None = DATA_ROOT / "data" / "test"
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = Path("../checkpoints") / "01_quartznet"

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

MUSAN_ROOT: Path | None = Path.home() / "datasets" / "musan" / "noise"
RIR_ROOT: Path | None = Path.home() / "datasets" / "RIRS_NOISES" / "simulated_rirs"
if MUSAN_ROOT is not None and not MUSAN_ROOT.exists():
    print(f"[warn] MUSAN not found at {MUSAN_ROOT} — AddNoise disabled")
    MUSAN_ROOT = None
if RIR_ROOT is not None and not RIR_ROOT.exists():
    print(f"[warn] RIR not found at {RIR_ROOT} — RIR disabled")
    RIR_ROOT = None

train=../notebooks/data/data/train, dev=../notebooks/data/data/dev, ckpts=../checkpoints/01_quartznet


## Шаг 1. Импорты

In [3]:
import json
import random
import time
import os

# Снижает фрагментацию VRAM — критично для torch.compile + переменных T.
# Должно быть выставлено до import torch (точнее до первой CUDA-аллокации).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

import torch
from torch.utils.data import DataLoader

from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.audio_aug_gpu import GPUAudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.quartznet import QuartzNet10x4
from gp1.text.vocab import CharVocab
from gp1.text.denormalize import words_to_digits
from gp1.train.trainer import Trainer, TrainerConfig
from gp1.train.optim import build_novograd
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import load_checkpoint
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.types import AugConfig

# cudnn.benchmark=False — переменные T после padding, autotune только мешает.
torch.backends.cudnn.benchmark = False
# TF32 для matmul — ускоряет fp32 операции на Ampere+ и Blackwell без потери CER.
torch.set_float32_matmul_precision("high")

# Limit torch.compile graph cache (dynamic=True can cache 50+ variants).
import torch._dynamo
torch._dynamo.config.cache_size_limit = 8

import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
# device must be constructed AFTER `import torch`.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

device=cuda


## Шаг 2. Гиперпараметры (FIXED + HP_GRID)

In [4]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 90,
    "grad_accum": 2,
    "subsample_factor": 2,
}

HP_GRID = {
    # --- model (QuartzNet 10x4 — arch fixed; only channel width & dropout vary) ---
    "d_model":                   [192, 256, 320],
    "dropout":                   [0.10, 0.15, 0.20],

    # --- optimizer / schedule ---
    "lr":                        [0.010, 0.015, 0.020],
    "weight_decay":              [1e-4, 1e-3],
    "warmup_steps":              [200, 500, 1000],
    "min_lr_ratio":              [0.01, 0.05],

    # --- SpecAugment (spectrogram) ---
    "specaug_freq_mask_param":   [10, 15, 20],
    "specaug_num_freq_masks":    [1, 2],
    "specaug_time_mask_param":   [20, 25, 35],
    "specaug_num_time_masks":    [2, 5, 8],
    "specaug_time_mask_max_ratio": [0.04, 0.05, 0.08],

    # --- Audio: speed & pitch (XOR — CPU path) ---
    "speed_prob":                [0.5, 1.0],
    "speed_factors":             [(0.9, 1.0, 1.1), (0.85, 1.0, 1.15)],
    "pitch_prob":                [0.0, 0.3, 0.5],
    "pitch_range_semitones":     [(-2.0, 2.0), (-3.0, 3.0)],

    # --- Audio: gain (CPU path) ---
    "gain_prob":                 [0.3, 0.7],
    "gain_db_range":             [(-4.0, 4.0), (-8.0, 8.0)],

    # --- Audio: GPU path (VTLP / AddNoise / RIR) ---
    "vtlp_prob":                 [0.0, 0.3, 0.5],
    "vtlp_alpha_range":          [(0.9, 1.1), (0.85, 1.15)],
    "noise_prob":                [0.0, 0.3],
    "noise_snr_db_range":        [(10.0, 20.0), (5.0, 20.0)],
    "rir_prob":                  [0.0, 0.1],
}

N_TRIALS = 8
SEED = 42
print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)


FIXED: {'samplerate': 16000, 'n_fft': 512, 'n_mels': 80, 'hop_length': 160, 'win_length': 400, 'max_epochs': 90, 'grad_accum': 2, 'subsample_factor': 2}
N_TRIALS: 8


## Шаг 3. Загрузка записей из CSV

In [5]:
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")

in_domain_speakers = {r.spk_id for r in train_records}
out_of_domain_count = sum(1 for r in dev_records if r.spk_id not in in_domain_speakers)
in_domain_count = sum(1 for r in dev_records if r.spk_id in in_domain_speakers)
print(f"dev speakers: in-domain={in_domain_count} samples, out-of-domain={out_of_domain_count} samples")


14:46:40 records_from_csv: loaded 12553 records from ../notebooks/data/data/train/train.csv
14:46:41 records_from_csv: loaded 2265 records from ../notebooks/data/data/dev/dev.csv
Train records: 12553, Dev records: 2265
dev speakers: in-domain=600 samples, out-of-domain=1665 samples


## Шаг 4. Vocabulary

In [6]:
vocab = CharVocab()
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")

Vocab size: 35, blank_id: 0


In [8]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)

for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()

preload audio: 100%|██████████| 14818/14818 [00:14<00:00, 991.36it/s] 

14:47:06 preload_audio_cache: 14818 tensors, 2.96 GB RAM


## Шаг 5. HP Random Search — Training Loop

In [ ]:
import gc
import math

SEED = 42
random.seed(SEED)
trial_results = []
run_root = CKPT_ROOT / f"01_quartznet_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 64
DL_WORKERS = 8  # speed/pitch perturb на CPU — pipeline нужны воркеры под GPU


def _worker_init(worker_id: int) -> None:
    """1 BLAS-тред на воркер + индивидуальный сид RNG аугментера.

    Без пересева у каждого воркера _augmenter._rng стартует с одного
    base_seed → одинаковые последовательности аугментаций (на разных
    сэмплах). PyTorch уже задаёт info.seed детерминированно от
    base_seed загрузчика и worker_id — используем его.
    """
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)

    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    aug_cfg = AugConfig(
        speed_factors=tuple(hp["speed_factors"]),
        speed_prob=hp["speed_prob"],
        pitch_prob=hp["pitch_prob"],
        pitch_range_semitones=tuple(hp["pitch_range_semitones"]),
        gain_prob=hp["gain_prob"],
        gain_db_range=tuple(hp["gain_db_range"]),
        seed=SEED + trial,
    )
    train_aug = AudioAugmenter(aug_cfg)
    gpu_aug = GPUAudioAugmenter(
        samplerate=FIXED["samplerate"],
        vtlp_prob=hp["vtlp_prob"],
        vtlp_alpha_range=tuple(hp["vtlp_alpha_range"]),
        noise_prob=hp["noise_prob"],
        noise_snr_db_range=tuple(hp["noise_snr_db_range"]),
        musan_root=MUSAN_ROOT,
        rir_prob=hp["rir_prob"],
        rir_root=RIR_ROOT,
    )
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=hp["specaug_num_freq_masks"],
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=hp["specaug_num_time_masks"],
        time_mask_max_ratio=hp["specaug_time_mask_max_ratio"],
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        augmenter=train_aug,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=3,
        worker_init_fn=_worker_init,
    )

    model = QuartzNet10x4(
        vocab_size=vocab.vocab_size,
        d_model=hp["d_model"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)
    model = torch.compile(model, mode="default", dynamic=True)

    ctc = CTCLoss(blank_id=vocab.blank_id)
    optimizer = build_novograd(
        model.parameters(),
        lr=hp["lr"],
        betas=(0.95, 0.5),
        weight_decay=hp["weight_decay"],
    )
    # math.ceil учитывает «хвостовой» optimizer.step(), который Trainer
    # делает при len(loader) % grad_accum != 0 — иначе scheduler уходит
    # в min_lr_ratio за несколько эпох до конца.
    steps_per_epoch = math.ceil(len(train_loader) / FIXED["grad_accum"])
    total_steps = steps_per_epoch * FIXED["max_epochs"]
    scheduler = build_cosine_warmup(
        optimizer,
        total_steps=total_steps,
        warmup_steps=hp["warmup_steps"],
        min_lr_ratio=hp["min_lr_ratio"],
    )

    trial_dir = run_root / f"trial_{trial:02d}"
    cfg = TrainerConfig(
        max_epochs=FIXED["max_epochs"],
        grad_accum=FIXED["grad_accum"],
        fp16_autocast=True,
        amp_dtype=torch.bfloat16,
        val_every_n_epochs=1,
        early_stop_patience=15,
        early_stop_metric="harmonic_in_out_cer",
        in_domain_speakers=in_domain_speakers,
        ckpt_dir=trial_dir,
    )
    trainer = Trainer(
        model=model,
        ctc_loss=ctc,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=dev_loader,
        vocab=vocab,
        config=cfg,
        device=device,
        audio_cfg={
            "n_fft": FIXED["n_fft"],
            "n_mels": FIXED["n_mels"],
            "hop_length": FIXED["hop_length"],
            "win_length": FIXED["win_length"],
            "samplerate": FIXED["samplerate"],
        },
        spec_augmenter=spec_aug,
        gpu_augmenter=gpu_aug,
    )
    result = trainer.fit()
    best_ckpt = result["best_ckpt_path"]
    trial_results.append({"trial": trial, "hp": hp, **result})

    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"trial {trial}: peak VRAM = {peak_gb:.2f} GB")

    if best_ckpt is not None:
        with open(trial_dir / "meta.json", "w") as f:
            json.dump(
                {
                    "arch": "quartznet_10x4",
                    "hparams": {**FIXED, **hp},
                    "val_cer": result["best_monitored"],
                    "trial": trial,
                },
                f,
                indent=2,
                default=str,
            )

    # Cleanup в конце trial.
    del trainer, model, optimizer, scheduler, ctc
    del train_loader, dev_loader, train_ds, dev_ds
    del train_aug, gpu_aug, spec_aug
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    torch._dynamo.reset()

print("\nHP search complete.")



=== Trial 1/8 ===
{
  "samplerate": 16000,
  "n_fft": 512,
  "n_mels": 80,
  "hop_length": 160,
  "win_length": 400,
  "max_epochs": 90,
  "grad_accum": 2,
  "subsample_factor": 2,
  "d_model": 320,
  "dropout": 0.1,
  "lr": 0.01,
  "weight_decay": 0.001,
  "warmup_steps": 200,
  "min_lr_ratio": 0.01,
  "specaug_freq_mask_param": 10,
  "specaug_num_freq_masks": 1,
  "specaug_time_mask_param": 35,
  "specaug_num_time_masks": 8,
  "specaug_time_mask_max_ratio": 0.08,
  "speed_prob": 0.5,
  "speed_factors": [
    0.85,
    1.0,
    1.15
  ],
  "pitch_prob": 0.0,
  "pitch_range_semitones": [
    -2.0,
    2.0
  ],
  "gain_prob": 0.3,
  "gain_db_range": [
    -4.0,
    4.0
  ],
  "vtlp_prob": 0.0,
  "vtlp_alpha_range": [
    0.9,
    1.1
  ],
  "noise_prob": 0.0,
  "noise_snr_db_range": [
    5.0,
    20.0
  ],
  "rir_prob": 0.0
}
10:29:14 GPUAudioAugmenter: loaded 930 files from /home/dench/datasets/musan/noise (29.5 MB)
10:29:30 GPUAudioAugmenter: loaded 60000 files from /home/dench/data

epochs:   0%|          | 0/90 [00:00<?, ?it/s]

epoch 1:   0%|          | 0/197 [00:00<?, ?it/s]

W0425 10:29:42.701000 22301 torch/_inductor/utils.py:1613] [0/0_1] Not enough SMs to use max_autotune_gemm mode


[Epoch 1/90] train  | loss=7.4807
[Epoch 1/90] val    | loss=5.3108  hm_cer=0.9962  (in=0.9960  out=0.9965)  best=0.9962  no_improve=0/15


...

[Epoch 3/90] train  | loss=2.7110
[Epoch 3/90] val    | loss=2.7871  hm_cer=0.8971  (in=0.8984  out=0.8959)  best=0.8971  no_improve=0/15


epoch 4:   0%|          | 0/197 [00:00<?, ?it/s]

[Epoch 4/90] train  | loss=2.6251
[Epoch 4/90] val    | loss=2.6511  hm_cer=0.8900  (in=0.8807  out=0.8995)  best=0.8900  no_improve=0/15


epoch 5:   0%|          | 0/197 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Шаг 6. Сводный отчёт трайлов

In [8]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_monitored"])
print(f"{'trial':>5} {'best_val_cer':>12} {'lr':>8} {'dropout':>8} {'d_model':>8}")
print("-" * 50)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_monitored']:>12.4f}"
        f" {hp['lr']:>8.4f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['d_model']:>8}"
    )

trial best_val_cer       lr  dropout  d_model
--------------------------------------------------
    0       0.1839   0.0200   0.1000      256
    7       0.2118   0.0200   0.1000      256
    3       0.3098   0.0150   0.1500      256
    4       0.3109   0.0200   0.2000      256
    6       0.3290   0.0200   0.2000      256
    1       0.3324   0.0100   0.1000      256
    5       0.4552   0.0100   0.1500      256
    2       0.5560   0.0100   0.2000      256
